In [2]:
import os
import json
from collections import defaultdict

def load_hero_registry(registry_path="data/1.json"):
    """
    步骤 1: 加载英雄名册，解析分路(roles)和ID映射
    """
    with open(registry_path, 'r', encoding='utf-8') as f:
        registry_data = json.load(f)
        
    id_to_name = {}
    name_to_info = {}
    
    for h in registry_data:
        h_id = str(h["id"])
        h_name = h["name"]
        
        # 将 "游走/对抗路" 切分为列表 ['游走', '对抗路']
        roles = [r.strip() for r in h["roles"].split('/') if r.strip()]
        
        id_to_name[h_id] = h_name
        name_to_info[h_name] = {
            "id": h_id,
            "roles": roles,
            "avatarUrl": h.get("avatarUrl", ""),
            "profession": h.get("profession", "")
        }
    return id_to_name, name_to_info


def load_hero_relations(heroes_dir, id_to_name):
    """
    步骤 2: 批量读取 data/heroes/{id}.json，建立快捷的关系矩阵
    """
    # 建立嵌套字典: relations[主英雄名][特定英雄名] = 分数
    synergy_matrix = defaultdict(lambda: defaultdict(float))
    counter_matrix = defaultdict(lambda: defaultdict(float))
    
    for h_id, h_name in id_to_name.items():
        file_path = os.path.join(heroes_dir, f"{h_id}.json")
        if not os.path.exists(file_path):
            continue
            
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        # 1. 提取与队友的协同分 (goodSynergies & badSynergies)
        for syn in data.get("goodSynergies", []):
            synergy_matrix[h_name][syn["heroName"]] = float(syn["synergyIndex"])
        for syn in data.get("badSynergies", []):
            synergy_matrix[h_name][syn["heroName"]] = float(syn["synergyIndex"])
            
        # 2. 提取与对手的克制分 (counters & counteredBy)
        # 统一使用 advantageIndex 作为标准。
        # counters 里为正数(克制对面)，counteredBy 里为负数(被对面克制)
        for cnt in data.get("counters", []):
            counter_matrix[h_name][cnt["heroName"]] = float(cnt["advantageIndex"])
        for cnt in data.get("counteredBy", []):
            counter_matrix[h_name][cnt["heroName"]] = float(cnt["advantageIndex"])
            
    return synergy_matrix, counter_matrix


def recommend_best_heroes(my_team, op_team, ban, id_to_name, name_to_info, synergy_matrix, counter_matrix):
    """
    步骤 3 & 4: 根据当前阵容，按分路计算并过滤推荐 Top 5
    """
    # 已经被两队挑选过的英雄组合，后续推荐需要排除
    picked_heroes = set(my_team + op_team + ban)
    
    # 用来按分路聚合候选人的字典
    role_recommendations = defaultdict(list)
    
    # 遍历 registry 中所有尚未被选走的英雄作为“候选人”
    for candidate_name, info in name_to_info.items():
        if candidate_name in picked_heroes:
            continue
            
        # 1. 计算与已知队友的协同总分
        total_synergy = 0.0
        for teammate in my_team:
            total_synergy += synergy_matrix[candidate_name].get(teammate, 0.0)
            
        # 2. 计算与已知敌人的克制总分（权重为 3）
        total_counter = 0.0
        for opponent in op_team:
            total_counter += counter_matrix[candidate_name].get(opponent, 0.0)
            
        # 3. 核心计算公式
        final_score = total_synergy + (total_counter * 3.0)
        
        # 4. 将候选人分类投喂到他所支持的每一个分路中
        for role in info["roles"]:
            role_recommendations[role].append({
                "name": candidate_name,
                "score": final_score,
                "id": info["id"],
                "profession": info["profession"]
            })
            
    # 5. 对每个分路内部按照得分从高到低排序，切片取 Top 10
    final_top5_by_role = {}
    all_standard_roles = ["对抗路", "中路", "发育路", "打野", "游走"]
    
    for role in all_standard_roles:
        candidates = role_recommendations.get(role, [])
        # 降序排序
        sorted_candidates = sorted(candidates, key=lambda x: x["score"], reverse=True)
        final_top5_by_role[role] = sorted_candidates[:10]
        
    return final_top5_by_role



# 1. 配置路径（根据你实际存放的文件夹修改）
REGISTRY_FILE = "data/1.json"
HEROES_DIR = "data/heroes"
id_to_name, name_to_info = load_hero_registry(REGISTRY_FILE)
synergy_matrix, counter_matrix = load_hero_relations(HEROES_DIR, id_to_name)
print("✅ 数据加载成功！开始实时阵容算分...\n")

✅ 数据加载成功！开始实时阵容算分...



In [6]:

current_my_team = "百里守约，墨子，元流之子（坦克）".split("，")
current_op_team = "少司缘，元歌，西施，孙尚香，云缨".split("，")
ban = "瑶".split("，")

valid_hero_names = set(name_to_info.keys())
# 收集输入错误的英雄
wrong_inputs = []
for hero in (current_my_team + current_op_team + ban):
    if hero not in valid_hero_names:
        wrong_inputs.append(hero)
        
# 🚨 触发熔断保护：如果存在打错的英雄，拒绝计算，直接抓出内鬼
if wrong_inputs:
    print("=" * 70)
    print(" ❌ 【名字输入错误！停止计算】 🚨")
    print("-" * 70)
    print(" 以下英雄无法在官方名册中找到，请检查是否打错了字或括号：")
    for wrong_name in wrong_inputs:
        print(f"   👉 '{wrong_name}'")
    print("\n 💡 提示：请核对是否多打了字、漏了字，或错用了英文括号。")
    print("=" * 70)


recommendations = recommend_best_heroes(
    my_team=current_my_team,
    op_team=current_op_team,
    ban=ban,
    id_to_name=id_to_name,
    name_to_info=name_to_info,
    synergy_matrix=synergy_matrix,
    counter_matrix=counter_matrix
)

# 3. 优雅地打印格式化终端看板
print("=" * 70)
print(f" 🎮  【BP 智能推荐看板】")
print(f" 🔵 我方阵容: {', '.join(current_my_team)}")
print(f" 🔴 敌方阵容: {', '.join(current_op_team)}")
print("=" * 70)

for role, top_list in recommendations.items():
    print(f"\n 📍 分路定位 —— 【{role}】 Top 10 最佳补位推荐:")
    print("-" * 55)
    if not top_list:
        print("   暂无推荐英雄")
        continue
    for rank, hero in enumerate(top_list, 1):
        # 打印出英雄、定位标签以及计算出的综合胜率分
        print(f"   Top {rank} | {hero['name']:<8} ({hero['profession']:<5}) | 综合推荐指数: {hero['score']:+.2f}")
print("=" * 70)


 ❌ 【名字输入错误！停止计算】 🚨
----------------------------------------------------------------------
 以下英雄无法在官方名册中找到，请检查是否打错了字或括号：
   👉 '元流之子（坦克）'

 💡 提示：请核对是否多打了字、漏了字，或错用了英文括号。
 🎮  【BP 智能推荐看板】
 🔵 我方阵容: 百里守约, 墨子, 元流之子（坦克）
 🔴 敌方阵容: 少司缘, 元歌, 西施, 孙尚香, 云缨

 📍 分路定位 —— 【对抗路】 Top 10 最佳补位推荐:
-------------------------------------------------------
   Top 1 | 钟无艳      (战野   ) | 综合推荐指数: +17.88
   Top 2 | 马超       (边核/野核) | 综合推荐指数: +11.38
   Top 3 | 关羽       (战边   ) | 综合推荐指数: +10.53
   Top 4 | 蒙恬       (坦边   ) | 综合推荐指数: +8.26
   Top 5 | 芈月       (开除   ) | 综合推荐指数: +6.61
   Top 6 | 孙策       (战边   ) | 综合推荐指数: +4.08
   Top 7 | 猪八戒      (坦野/坦边) | 综合推荐指数: +3.22
   Top 8 | 老夫子      (战边   ) | 综合推荐指数: +0.94
   Top 9 | 蚩奼       (边核/带线射) | 综合推荐指数: +0.92
   Top 10 | 花木兰      (战边   ) | 综合推荐指数: +0.82

 📍 分路定位 —— 【中路】 Top 10 最佳补位推荐:
-------------------------------------------------------
   Top 1 | 上官婉儿     (大法核  ) | 综合推荐指数: +31.14
   Top 2 | 高渐离      (大法核  ) | 综合推荐指数: +20.66
   Top 3 | 张良       (怕开团辅 ) | 综合推荐指数: +18.60
  